In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# COMMAND ----------

def write_to_delta(
    input_df,
    target_table,
    merge_condition,
    columns_to_update,
    match_condition=None
):
    """
    Single shared upsert helper used by both silver and gold layers.

    Parameters
    ----------
    input_df        : DataFrame to persist.
    target_table    : Fully-qualified Delta table name  (catalog.schema.table).
    merge_condition : JOIN predicate between target (t) and source (s).
    columns_to_update : List of column names to overwrite on a matched row.
    match_condition : Optional extra predicate evaluated *after* a row matches,
                      e.g. "s.batch_id >= t.batch_id" to prevent stale overwrites.
                      When None, every matched row is updated unconditionally.
    """
    final_df = (
        input_df
        .withColumn("created_timestamp", F.current_timestamp())
        .withColumn("updated_timestamp", F.current_timestamp())
    )

    if not spark.catalog.tableExists(target_table):
        (
            final_df.write
                .format("delta")
                .mode("overwrite")
                .saveAsTable(target_table)
        )
    else:
        delta_table = DeltaTable.forName(spark, target_table)
        update_map = {col: f"s.{col}" for col in columns_to_update}
        update_map["updated_timestamp"] = "s.updated_timestamp"

        merged = (
            delta_table.alias("t")
            .merge(final_df.alias("s"), merge_condition)
        )

        if match_condition is not None:
            merged = merged.whenMatchedUpdate(condition=match_condition, set=update_map)
        else:
            merged = merged.whenMatchedUpdate(set=update_map)

        merged.whenNotMatchedInsertAll().execute()


In [0]:
def write_to_silver(input_df, target_table, merge_condition, columns_to_update):

    write_to_delta(
        input_df=input_df,
        target_table=target_table,
        merge_condition=merge_condition,
        columns_to_update=columns_to_update,
        match_condition="s.batch_id >= t.batch_id"
    )